In [7]:
# ============================================================
# ERIP - ENTERPRISE RISK INTELLIGENCE PLATFORM
# ============================================================
#
# Notebook
# --------
# nb_build_macro_dimension
#
# Layer
# -----
# Silver Layer
#
# Purpose
# -------
# Build the enterprise Macroeconomic Scenario Dimension from
# the Bronze Macroeconomic Data table.
#
# Business Objective
# ------------------
# Create a standardized macroeconomic scenario entity used for:
#
# • Stress Testing
# • Scenario Analysis
# • IFRS 9
# • Basel III Capital Planning
# • Expected Credit Loss (ECL)
# • AI Decision Intelligence
#
# Enterprise Concepts
# -------------------
# ✓ Medallion Architecture
# ✓ Analytics Engineering
# ✓ Enterprise Scenario Modelling
# ✓ Stress Testing
# ✓ Basel III
# ✓ IFRS 9
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

# ------------------------------------------------------------
# Source / Target Configuration
# ------------------------------------------------------------

source_table = "bronze_macroeconomic_data"
target_table = "silver_macro"
pipeline_name = "nb_build_macro_dimension"

# Pipeline Execution Timestamp

run_start_time = datetime.now()

print("ERIP Silver Macro Dimension Build Started")

StatementMeta(, 9b21cf7c-97cd-47ae-b52f-873c745f737d, 9, Finished, Available, Finished, False)

ERIP Silver Macro Dimension Build Started


In [8]:
# ============================================================
# SECTION 2 - READ BRONZE MACROECONOMIC TABLE
# ============================================================
#
# Purpose
# -------
# Read the validated Bronze Macroeconomic Scenario table.
#
# Note
# ----
# The Silver layer always consumes governed Bronze Delta tables.
#
# Enterprise Concepts
# -------------------
# ✓ Delta Lake
# ✓ Data Lineage
# ✓ Medallion Architecture
# ============================================================

macro_bronze_df = spark.table(source_table)

display(macro_bronze_df.limit(10))

print(f"Rows read : {macro_bronze_df.count()}")
print(f"Columns   : {len(macro_bronze_df.columns)}")

StatementMeta(, 9b21cf7c-97cd-47ae-b52f-873c745f737d, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cf13ce5e-a10f-457b-ac77-25cb5326b582)

Rows read : 108
Columns   : 17


In [9]:
# ============================================================
# SECTION 3 - STANDARDIZE MACROECONOMIC DIMENSION
# ============================================================
#
# Purpose
# -------
# Standardize the Bronze Macroeconomic Scenario table into
# an enterprise-ready Scenario Dimension.
#
# Standardization Activities
# --------------------------
# • Trim whitespace
# • Standardize text case
# • Convert numeric measures
# • Convert scenario month to Date
# • Remove duplicate scenarios
#
# Enterprise Concepts
# -------------------
# ✓ Data Standardization
# ✓ Business Entity
# ✓ Scenario Modelling
# ✓ Enterprise Analytics
# ============================================================

silver_macro_df = (
    macro_bronze_df
    .select(
        upper(trim(col("scenario_id"))).alias("scenario_id"),
        initcap(trim(col("scenario_name"))).alias("scenario_name"),
        initcap(trim(col("scenario_severity"))).alias("scenario_severity"),
        to_date(col("month")).alias("scenario_month"),
        col("gdp_growth_pct").cast("double").alias("gdp_growth_pct"),
        col("inflation_pct").cast("double").alias("inflation_pct"),
        col("interest_rate_pct").cast("double").alias("interest_rate_pct"),
        col("unemployment_pct").cast("double").alias("unemployment_pct"),
        col("commercial_property_price_index").cast("double").alias("commercial_property_price_index"),
        col("credit_spread_bps").cast("double").alias("credit_spread_bps"),
        col("pd_stress_multiplier").cast("double").alias("pd_stress_multiplier"),
        col("lgd_stress_multiplier").cast("double").alias("lgd_stress_multiplier"),
        current_timestamp().alias("silver_updated_timestamp")
    )
    .dropDuplicates(["scenario_id"])
)

StatementMeta(, 9b21cf7c-97cd-47ae-b52f-873c745f737d, 11, Finished, Available, Finished, False)

In [10]:
# ============================================================
# SECTION 4 - BUSINESS ENRICHMENT
# ============================================================
#
# Purpose
# -------
# Create derived macroeconomic attributes required for
# enterprise stress testing and scenario analysis.
#
# Derived Attributes
# ------------------
# • Scenario Surrogate Key
# • Scenario Rank
# • Economic Cycle
# • Interest Rate Environment
# • Stress Intensity
#
# Enterprise Concepts
# -------------------
# ✓ Basel III
# ✓ IFRS 9
# ✓ Stress Testing
# ✓ Scenario Analysis
# ✓ Enterprise Risk Analytics
# ✓ Dimensional Modelling
# ============================================================

window_spec = Window.orderBy("scenario_month","scenario_name")

silver_macro_df = (
    silver_macro_df

    .withColumn("macro_sk", row_number().over(window_spec))

    .withColumn(
        "scenario_rank",
        when(col("scenario_name")=="Baseline",1)
        .when(col("scenario_name")=="Adverse",2)
        .when(col("scenario_name")=="Severe",3)
        .otherwise(99)
    )

    .withColumn(
        "economic_cycle",
        when(col("gdp_growth_pct")<0,"Contraction")
        .when(col("gdp_growth_pct")<1.5,"Weak Growth")
        .when(col("gdp_growth_pct")<3,"Moderate Growth")
        .otherwise("Strong Growth")
    )

    .withColumn(
        "interest_rate_environment",
        when(col("interest_rate_pct")<2,"Low Rate")
        .when(col("interest_rate_pct")<5,"Normal Rate")
        .otherwise("High Rate")
    )

    .withColumn(
        "stress_intensity",
        when(col("scenario_name")=="Severe","High Stress")
        .when(col("scenario_name")=="Adverse","Moderate Stress")
        .otherwise("Base Case")
    )

    .select(
        "macro_sk",
        "scenario_id",
        "scenario_name",
        "scenario_rank",
        "scenario_severity",
        "scenario_month",
        "gdp_growth_pct",
        "inflation_pct",
        "interest_rate_pct",
        "unemployment_pct",
        "commercial_property_price_index",
        "credit_spread_bps",
        "pd_stress_multiplier",
        "lgd_stress_multiplier",
        "economic_cycle",
        "interest_rate_environment",
        "stress_intensity",
        "silver_updated_timestamp"
    )
)

display(silver_macro_df.limit(10))

StatementMeta(, 9b21cf7c-97cd-47ae-b52f-873c745f737d, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 46616eef-ef7d-44fd-b3b7-c1415032671b)

In [11]:
# ============================================================
# SECTION 5 - SILVER DATA QUALITY VALIDATION
# ============================================================
#
# Purpose
# -------
# Validate the Silver Macroeconomic Dimension before publishing.
#
# Validation Checks
# -----------------
# • Duplicate Scenario IDs
# • Null Scenario IDs
# • Null Surrogate Keys
# • Invalid PD Stress Multiplier
# • Invalid LGD Stress Multiplier
#
# Enterprise Concepts
# -------------------
# ✓ Data Quality
# ✓ Data Governance
# ✓ Stress Testing Controls
# ✓ Trusted Analytics Layer
# ============================================================

total_rows = silver_macro_df.count()

duplicate_scenario_ids = total_rows - silver_macro_df.select("scenario_id").distinct().count()

null_scenario_ids = silver_macro_df.filter(col("scenario_id").isNull()).count()

null_surrogate_keys = silver_macro_df.filter(col("macro_sk").isNull()).count()

invalid_pd_multiplier = silver_macro_df.filter(col("pd_stress_multiplier") <= 0).count()

invalid_lgd_multiplier = silver_macro_df.filter(col("lgd_stress_multiplier") <= 0).count()

print("Silver Macro Quality Checks")
print("---------------------------")
print(f"Rows: {total_rows}")
print(f"Duplicate Scenario IDs: {duplicate_scenario_ids}")
print(f"Null Scenario IDs: {null_scenario_ids}")
print(f"Null Surrogate Keys: {null_surrogate_keys}")
print(f"Invalid PD Multiplier: {invalid_pd_multiplier}")
print(f"Invalid LGD Multiplier: {invalid_lgd_multiplier}")

if (
    duplicate_scenario_ids > 0 or
    null_scenario_ids > 0 or
    null_surrogate_keys > 0 or
    invalid_pd_multiplier > 0 or
    invalid_lgd_multiplier > 0
):
    raise Exception("Silver Macro Validation Failed")
else:
    print("✓ Silver Macro Validation Passed")

StatementMeta(, 9b21cf7c-97cd-47ae-b52f-873c745f737d, 13, Finished, Available, Finished, False)

Silver Macro Quality Checks
---------------------------
Rows: 108
Duplicate Scenario IDs: 0
Null Scenario IDs: 0
Null Surrogate Keys: 0
Invalid PD Multiplier: 0
Invalid LGD Multiplier: 0
✓ Silver Macro Validation Passed


In [12]:
# ============================================================
# SECTION 6 - WRITE SILVER DELTA TABLE
# ============================================================
#
# Purpose
# -------
# Publish the enterprise Macroeconomic Scenario Dimension to
# the Silver Layer.
#
# Output
# ------
# silver_macro
#
# Enterprise Concepts
# -------------------
# ✓ Delta Lake
# ✓ Analytics Engineering
# ✓ Enterprise Risk Platform
# ============================================================

silver_macro_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(target_table)

print(f"✓ Silver table created: {target_table}")
print(f"Rows written: {silver_macro_df.count()}")

StatementMeta(, 9b21cf7c-97cd-47ae-b52f-873c745f737d, 14, Finished, Available, Finished, False)

✓ Silver table created: silver_macro
Rows written: 108
